# Intel Image Classification: a CNN from scratch

This notebook follows the order of **Lab 3** (Code 1 to 12), applied to our project:
RGB images resized to 128x128 and 6 classes (buildings, forest, glacier, mountain, sea, street).

**Before running**
1. Run `python -m src.setup_dataset` once (downloads and cleans the data).
2. Start the notebook from the **project root** so `import src...` works.
3. Run the cells in order, each one **only once**.

**Lab rule:** before each cell, say out loud what you expect it to print, then run it and compare.

## Code 1: Imports and setup
- `SEED` fixes the randomness so results are roughly repeatable.
- We check that the notebook is running from the project root.
- We print whether a GPU is available. Training works without one, just slower.

In [ ]:
import os
import random

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

assert os.path.isdir("src"), "Run this notebook from the project root (the folder containing src/)."

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## Code 2: Load the data
The lab used `fashion_mnist.load_data()`. Our images live in folders, so we call `load_datasets()` from `src/preprocessing.py`. It:
- reads the images and resizes them to 128x128 RGB,
- splits the training folder into 80% train and 20% validation (fixed seed),
- keeps the test set separate and unshuffled,
- normalizes pixels to 0-1 and augments the training images only.

**Predict:** how many batches in each dataset? (11,205 / 32, 2,801 / 32, 2,997 / 32, rounded up)

In [ ]:
from src.preprocessing import load_datasets, IMAGE_SIZE, BATCH_SIZE

train_ds, val_ds, test_ds, class_names = load_datasets()
NUM_CLASSES = len(class_names)

print("Classes:", class_names)
print("Batches per epoch (train / val / test):", len(train_ds), len(val_ds), len(test_ds))

## Code 3: Look at the images
We show validation images (no augmentation) with their labels, to confirm images and labels match before any training.

In [ ]:
images, labels = next(iter(val_ds))

plt.figure(figsize=(12, 6))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(images[i])          # RGB in [0, 1]: no cmap, no squeeze needed
    plt.title(class_names[int(labels[i])])
    plt.axis("off")
plt.tight_layout()
plt.show()

### Images per class
Check that the classes are roughly balanced. A very small class gets ignored by the model.

In [ ]:
def class_counts(ds):
    counts = np.zeros(NUM_CLASSES, dtype=int)
    for _, y in ds:
        counts += np.bincount(y.numpy(), minlength=NUM_CLASSES)
    return counts

for name, ds in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    c = class_counts(ds)
    print(f"{name:5s} total={c.sum():5d}  ", dict(zip(class_names, c.tolist())))

## Code 4: Check the preprocessing
In the lab this step was `/255.0` and `np.newaxis`. Here `load_datasets` already did both, so we only verify the result:
- shape `(batch, 128, 128, 3)`: the 3 RGB channels come for free,
- dtype `float32`, values between 0 and 1.

In [ ]:
x, y = next(iter(train_ds))
print("Image batch shape:", x.shape)
print("Labels shape:", y.shape)
print("dtype:", x.dtype)
print("min / max:", float(tf.reduce_min(x)), float(tf.reduce_max(x)))

### What the augmentation does
We take one image and augment it five times. **Predict:** do the results still look realistic? (rotation up to about 29 degrees, zoom, horizontal flip. If the rotation looks too strong, that is a good first experiment.)

In [ ]:
from src.augmentation import build_augmentation

aug = build_augmentation()
sample, _ = next(iter(val_ds))
sample = sample[:1]

plt.figure(figsize=(15, 3))
plt.subplot(1, 6, 1)
plt.imshow(sample[0]); plt.title("Original"); plt.axis("off")
for i in range(5):
    plt.subplot(1, 6, i + 2)
    plt.imshow(tf.clip_by_value(aug(sample, training=True)[0], 0, 1))
    plt.title(f"Aug {i + 1}"); plt.axis("off")
plt.tight_layout()
plt.show()

## Code 5: Build the CNN
Same idea as the lab (Conv -> ReLU -> Pool, then the classifier) with three changes because our images are bigger (128x128x3):
1. **Three Conv blocks** (32 / 64 / 128) instead of two.
2. **GlobalAveragePooling2D** instead of Flatten: it averages each feature map, giving 128 numbers instead of 25,088. That cuts the parameters from about 1.7M to about 0.1M and reduces overfitting.
3. **Dropout(0.3)** before the output layer.

**Shape math** (3x3 conv with no padding: size - 2, pooling halves, rounded down):

| Layer | Output shape | Parameters |
|---|---|---|
| Input | 128x128x3 | 0 |
| Conv 32 | 126x126x32 | 896 |
| Pool | 63x63x32 | 0 |
| Conv 64 | 61x61x64 | 18,496 |
| Pool | 30x30x64 | 0 |
| Conv 128 | 28x28x128 | 73,856 |
| Pool | 14x14x128 | 0 |
| GlobalAvgPool | 128 | 0 |
| Dense 64 | 64 | 8,256 |
| Dense 6 | 6 | 390 |

Expected total: **101,894**. Compare it with `model.summary()`.

In [ ]:
from tensorflow.keras import layers


def build_model(input_shape=(*IMAGE_SIZE, 3), num_classes=NUM_CLASSES, dropout=0.3):
    return tf.keras.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.GlobalAveragePooling2D(),
        layers.Dense(64, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(num_classes, activation="softmax"),
    ], name="intel_cnn")


model = build_model()
model.summary()

## Code 6: Compile
- `adam`: the optimizer that updates the weights.
- `sparse_categorical_crossentropy`: our labels are integers (`label_mode="int"`) and the output is softmax.
- `accuracy`: for monitoring only.

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

## Code 7: Train
- `train_ds` already yields batches, so no `batch_size` or `validation_split` is needed.
- `validation_data=val_ds` is used for monitoring after every epoch.
- `epochs=30` is only a ceiling. **EarlyStopping** stops when `val_loss` has not improved for 5 epochs and restores the best weights.
- **ModelCheckpoint** saves the best model to `models/best_model.keras` (the folder is git-ignored).

Note that training `accuracy` can be lower than `val_accuracy` early on, because training numbers are measured with augmentation and Dropout switched on.

In [ ]:
os.makedirs("models", exist_ok=True)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "models/best_model.keras", monitor="val_loss", save_best_only=True
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks,
)

## Code 8: Learning curves
- Both curves improve together: healthy learning.
- Training improves while validation stalls or gets worse and the gap widens: **overfitting**.
- Both are poor: underfitting.

The dashed line marks the best epoch.

In [ ]:
hist = history.history
best_epoch = int(np.argmin(hist["val_loss"]))

plt.figure(figsize=(12, 4))
for i, (metric, title) in enumerate([("accuracy", "Accuracy"), ("loss", "Loss")], start=1):
    plt.subplot(1, 2, i)
    plt.plot(hist[metric], label="Training")
    plt.plot(hist["val_" + metric], label="Validation")
    plt.axvline(best_epoch, color="gray", linestyle="--", label="Best epoch")
    plt.title(f"{title} over epochs")
    plt.xlabel("Epoch"); plt.ylabel(title)
    plt.legend()
plt.tight_layout()
plt.show()

print("Best epoch (0-based):", best_epoch)
print("Val accuracy at best epoch:", round(hist["val_accuracy"][best_epoch], 4))

## Code 9: Evaluate on the test set
**Once, at the very end.** If you change the model because of this number, it becomes a validation score and loses its value.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.2%}")

## Code 10: Predictions and the confusion matrix
`test_ds` is not shuffled (`shuffle=False`), so predictions line up with the labels. We gather labels and images into arrays, then:
- `predict` returns probabilities of shape `(N, 6)`,
- `argmax` picks the highest-probability class,
- rows = actual class, columns = predicted class, diagonal = correct.

Look at the accuracy of **each class**, not only the overall number.

In [ ]:
X_test = np.concatenate([x.numpy() for x, _ in test_ds])
y_true = np.concatenate([y.numpy() for _, y in test_ds])

probabilities = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
y_pred = np.argmax(probabilities, axis=1)

# Sanity check: must match model.evaluate above.
print("Accuracy from predictions:", f"{(y_pred == y_true).mean():.2%}")

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    ax=ax, xticks_rotation=45, cmap="Blues"
)
plt.show()

print("\nPer-class accuracy:")
for name, acc in zip(class_names, cm.diagonal() / cm.sum(axis=1)):
    print(f"  {name:10s} {acc:.1%}")

pairs = [
    (cm[i, j], class_names[i], class_names[j])
    for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i != j
]
print("\nMost frequent mistakes (actual -> predicted):")
for n, a, b in sorted(pairs, reverse=True)[:5]:
    print(f"  {n:4d}  {a} -> {b}")

## Code 11: Show predictions
Ten random test images. Green is correct, red is wrong. (No `squeeze` needed because the images are RGB.)

In [ ]:
rng = np.random.default_rng(SEED)
idx = rng.choice(len(X_test), size=10, replace=False)

plt.figure(figsize=(15, 6))
for k, i in enumerate(idx):
    plt.subplot(2, 5, k + 1)
    plt.imshow(X_test[i])
    true, pred = class_names[y_true[i]], class_names[y_pred[i]]
    plt.title(f"True: {true}\nPred: {pred}",
              color="green" if true == pred else "red", fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

## Code 12: Error analysis
We show misclassified images with the model's confidence, and print the probabilities of the first mistake:
- if two probabilities are close (for example 0.50 and 0.48), the model was unsure rather than confidently wrong,
- ask yourself: could the image honestly be two classes? Which pair keeps coming up?

In [ ]:
wrong = np.where(y_pred != y_true)[0]
print(f"Mistakes: {len(wrong)} of {len(y_true)}")

plt.figure(figsize=(15, 6))
for k, i in enumerate(wrong[:10]):
    plt.subplot(2, 5, k + 1)
    plt.imshow(X_test[i])
    conf = probabilities[i, y_pred[i]]
    plt.title(f"True: {class_names[y_true[i]]}\nPred: {class_names[y_pred[i]]} ({conf:.0%})",
              color="red", fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

first = wrong[0]
print("\nProbabilities for the first mistake:")
for name, p in zip(class_names, probabilities[first]):
    print(f"  {name:10s}: {p:.3f}")

## Experiments (Lab challenge, slide 28)
Change **one thing** at a time and record what actually happened, even if it is worse. Rebuild the model from Code 5 before each run.

| # | Change | Params | Best val acc | Test acc | Train-val gap | Notes |
|---|---|---|---|---|---|---|
| 0 | Baseline (32/64/128 + Dropout 0.3) | 101,894 | | | | |
| 1 | `dropout=0.5` | 101,894 | | | | |
| 2 | Filters 16/32/64 | | | | | |
| 3 | Filters 64/128/256 | | | | | |
| 4 | No augmentation: `load_datasets(apply_augmentation=False)` | 101,894 | | | | |
| 5 | A fourth Conv block | | | | | |

**Report questions:** which two classes get confused most often, and did they improve after the experiments?